# Fraud Detection in Electricity and Gas Consumption

Team project for the Zindi challenge
[Fraud Detection in Electricity and Gas Consumption](https://zindi.africa/competitions/fraud-detection-in-electricity-and-gas-consumption-challenge).

**Goal:** find customers who cheat with their electricity or gas meter.

We work through 7 stages, from "understand the problem" to "does the model really help?":

1. Data Ingestion & Problem Framing         -- load the data, decide what we want to predict
2. Train/Test Split & EDA Light             -- put test data aside, then explore the rest
3. Cleaning & Feature Preprocessing         -- fix errors, build useful columns
4. Baselines & Feature Selection            -- the simplest solution, as a score to beat
5. Model Exploration & CV                   -- try several models and test them fairly
6. Diagnostics, Plots & Interpretability    -- understand why the model decides
7. Business Alignment & Turning Points      -- is it worth it for STEG?

- **Status:** ✅ done · 🔄 in progress · ⏳ open
- **In every stage:** "In short" at the top, decisions in `### Decision: …` cells, results in the **Stage End** block.
- **Names:** `client_train` / `invoice_train` = full data (Stage 1 only); `train_clients` / `train_invoices` = our train part; `holdout_*` = untouched until Stage 7 (see *Naming* in Stage 2).

### Results so far

| Stage | Result |
| --- | --- |
| 1 Data & framing | binary classification per client, 5.6 % fraud, metrics ROC-AUC + Recall/F2 |
| 2 Split & EDA | 20 % holdout by client; decision per column |
| 3 Features | one row per client, 18 features; pipeline median → log1p → scaler |
| 4 Baselines | Dummy 0.500 · Scorecard 0.670 · **Logistic Regression 0.788** |

**Features in the baseline (17):**

| Group | Features |
| --- | --- |
| Client | `catg_51` |
| Bills & counters | `n_invoices`, `n_counters`, `history_years` |
| Consumption | `level_1_mean` … `level_4_mean`, `total_mean`, `total_std`, `zero_share`, `total_per_month_mean` |
| Flags | `months_error`, `mismatch_any`, `tarif_not_11`, `statue_1`, `statue_other` |

**Later (Stage 5):** `region`, `disrict`, seasonal anomaly (Hadi), years as client, `counter_code`.
**Dropped:** `client_id`, `counter_number` (IDs), `creation_date`, `reading_remarque`, `counter_type`, `counter_coefficient`, `total_max`.

# ✅ Stage 1: Data Ingestion & Problem Framing

### In short

**Problem:** some customers manipulate their electricity or gas meter. STEG loses money but cannot inspect everyone → we **rank the customers by fraud risk**, so the inspection teams visit the most suspicious ones first.

- **Data:** `client_train` (135,493 clients, with `target` = fraud yes/no) and `invoice_train` (4.48 M bills), linked by `client_id` (one client has ~33 bills).
- **Task:** binary classification (two groups: fraud / no fraud). Only 5.6 % commit fraud, so accuracy is useless – always saying "no fraud" is already 94.4 % right.
- **Metrics:** **ROC-AUC** (Zindi: does the model put fraud clients higher on the list than honest ones?) and **Recall & F2** (business: how many of the fraud cases do we find?).

**→ Next:** put test data aside and explore the rest.
### Data

We get two kinds of tables:

| File | One row is … | What's inside |
| --- | --- | --- |
| `client_train.csv` / `client_test.csv` | one client | where the client lives (district, region), client type, creation date, **target** = fraud yes/no (train only) |
| `invoice_train.csv` / `invoice_test.csv` | one bill (invoice) | date, tariff, meter info, consumption in 4 levels, meter readings, meter type (ELEC = electricity, GAZ = gas) |

The Zindi test files have **no answers** (no `target`). We can only send our predictions to Zindi.
To check our model ourselves, we keep a part of `client_train` aside (our holdout).

## 1.1 Loading and overview the data (first look)

In [ ]:
# Imports
from pathlib import Path
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# Load the raw tables
client_train = pd.read_csv("data/train/client_train.csv")
invoice_train = pd.read_csv("data/train/invoice_train.csv", low_memory=False)
client_test = pd.read_csv("data/test/client_test.csv")
invoice_test = pd.read_csv("data/test/invoice_test.csv", low_memory=False)
submission = pd.read_csv("data/output/submission.csv")

for name, df in {
    "client_train": client_train,
    "invoice_train": invoice_train,
    "client_test": client_test,
    "invoice_test": invoice_test,
    "submission": submission,
}.items():
    print(f"{name:14s} {df.shape}")


In [ ]:
# First look: client tables
print(client_train.head())
print(client_test.head(10))
display(client_train.describe().T)
# print(client_train["target"].value_counts().sum())
print("This is the fraud percantage")
print(client_train["target"].mean()*100)

In [ ]:
# First look: invoice table
invoice_train.head(21)
client_train.columns

invoice_train.columns

In [ ]:
# Missing and zero values
print("Do we have NaN-values?: ")
print(invoice_train.isna().sum())

print("Do we have Values, that are 'zero'?:")
print((invoice_train == 0).sum())

## 1.2 Column overview

In [ ]:
# Column overview: client table
def overview(df):
    return pd.DataFrame({
        "dtype": df.dtypes,
        "n_unique": df.nunique(),
        "n_missing": df.isna().sum(),
        "example": df.iloc[0],
    })


overview(client_train)


In [ ]:
# Column overview: invoice table
overview(invoice_train)

## 1.3 Join key and data grain

### What does it mean?

- **Data grain** = what one row means. In `client_train` one row is **one client**, in `invoice_train` one row is **one bill**.
- **Join key** = the column that connects both tables: `client_id`.
- One client has **many** bills (33 on average). Before we combine the tables, we have to summarise the bills per client (e.g. average consumption). Otherwise every client would show up ~33 times – and their `target` too.

In [ ]:
# Join key and data grain
print("Each client appears only once:", client_train["client_id"].is_unique)
print("Invoices without a known client:", (~invoice_train["client_id"].isin(client_train["client_id"])).sum())
print("Clients without any invoice:", (~client_train["client_id"].isin(invoice_train["client_id"])).sum())

# How many invoices does one client have?
invoice_train.groupby("client_id").size().describe()

## 1.4 Target distribution

In [ ]:
# Target distribution
print(client_train["target"].value_counts(dropna=False))
print()
print(client_train["target"].value_counts(normalize=True).round(3)*100)

client_train["target"].value_counts().plot.bar(title="Target distribution (0 = no fraud, 1 = fraud)")
plt.show()


# ✅ Stage 1 End: Results & Learnings (Marker)

## 1.5 Problem Framing

### Stakeholder, goal and metric

**Who needs this?** STEG, the Tunisian electricity and gas company – especially the teams that check customers on site.

**Problem:** Some customers manipulate their meter so it shows less than they really use. STEG loses a lot of money.
Checking a customer on site is expensive, so STEG can't check everyone.

**Goal:** Give every customer a fraud probability (how likely they cheat). Then the teams visit the most suspicious customers first.

**ML task:** Binary classification (sorting into two groups) per client.
- Target `y`: `target` in `client_train.csv` – 1 = fraud, 0 = no fraud, one answer per client
- Information: ~4.5 million bills in `invoice_train.csv`, summarised per `client_id`
- Imbalance: only 5.6 % of the clients commit fraud

**How we measure success:**
- **Zindi: ROC-AUC.** Checks if the model puts fraud clients higher on the list than honest ones. 1.0 = perfect order, 0.5 = random guessing. No threshold needed.
- **Business: Recall & F2.** Recall = how many of all fraud clients we actually find. A missed fraud costs money, so recall matters most.
  F2 also looks at precision (how many flagged clients really cheat), but counts recall more – so "flag everyone" does not win.
- **Accuracy is useless here:** a model that always says "no fraud" is right 94.4 % of the time – and finds zero fraud.

### Rubber-duck checklist

- [x] **Do we know exactly what the target $y$ is?** → Yes: `target` in `client_train.csv`, one answer per client.
- [x] **Regression, classification, clustering or anomaly detection?** → Binary classification: two groups (fraud / no fraud). The model gives a probability, so we can rank the clients.
- [x] **Do all features exist at prediction time?** → Not fully checkable. We don't know **when** a fraud was found. If STEG changed the meter or corrected bills **after** finding the fraud, those bills already "know" the answer (data leakage = the model sees information it would not have in real life). Checked in Stage 2 (end of the bill history) and again in Stage 6 (features that are suspiciously strong).

## 1.6 Learnings (Marker)

### Stage 1: Marker – Problem type first, then algorithm and metric

Three separate questions, in this order:

1. **Problem type:** What should come out? This only depends on what $y$ looks like – or if there is a $y$ at all.
2. **Algorithm:** Which method calculates it? (e.g. Decision Tree, LightGBM)
3. **Metric:** How do we measure if it is good? (e.g. Accuracy, Recall, RMSE)

| Problem type | $y$ looks like | Question | Example metric | Example algorithm |
| --- | --- | --- | --- | --- |
| **Regression** | a number, e.g. 1,234 kWh | How much? | RMSE, MAE | Linear Regression, Decision Tree |
| **Classification** | a class, e.g. 0/1 | Which group? | Accuracy, Recall, Precision, F1, AUC | Logistic Regression, Decision Tree |
| **Clustering** | **no $y$** | Which clients are similar? | Silhouette score | k-Means |
| **Anomaly detection** | **no $y$** (or almost none) | Who is unusual? | Precision@k | Isolation Forest |

- Accuracy is for classification. For regression you measure how far off the predicted number is (RMSE).
- A Decision Tree is a method, not a problem type. It can do regression and classification, but not clustering.
- Recall and Precision count right and wrong guesses – so you need the true answers (labels).
- Fraud is often anomaly detection – but only when nobody knows who cheated. Here we know it, so it is classification.

# ✅ Stage 2: Data Splitting & EDA Light

### In short

We put a 20 % **holdout** aside – our "final exam", untouched until Stage 7. It is split by client and **stratified** (same 5.6 % fraud in both parts). All analysis uses only the train part, so no information from the exam leaks into our decisions (**data leakage**).

- **No missing values**, but impossible values: bills covering more than 240 months, bills dated before the client existed.
- **Fraud signals:** client category 51 (16.9 % fraud vs. 5.6 % on average), several meters per client (`n_counters`), bills where the consumption does not match the meter readings (`mismatch`), tariff other than 11, counter status 1 (less fraud).
- Every column got a keep / drop decision with a reason (Stage 2 End).

**→ Next:** turn these decisions into one row per client.
### Split strategy

Before we look deeper into the data, we put 20 % aside as a "final exam" (holdout).

- **What we split:** the clients in `client_train` (one row = one client = one answer). Their bills go with them via `client_id`, so all bills of a client end up on the same side.
- **Why not split the bills?** Then the same client would be in train and holdout. The model would just recognise the client instead of learning what fraud looks like (data leakage).
- **Stratified (`stratify=y`):** both parts get the same share of fraud (5.6 %).
- **No time split:** a time split is needed when you predict the future. Our `target` has no date (just "fraud yes/no" per client), so a random split is fine.
- **Holdout (20 %):** stays untouched until Stage 7. Everything below uses the train part only.

### Naming

| Name | What | Use it in |
| --- | --- | --- |
| `client_train`, `invoice_train` | **full** Zindi training data | Stage 1 only, before the split |
| `train_clients`, `train_invoices` | **our train part** (80 %) | from Stage 2 on: EDA, features, models |
| `holdout_clients`, `holdout_invoices` | **our holdout** (20 %) | only at the very end (Stage 7) |

## 2.1 Train/holdout split at client level

In [ ]:
# Stratified train/holdout split at client level
from sklearn.model_selection import train_test_split

train_clients, holdout_clients = train_test_split(
    client_train,
    test_size=0.2,
    stratify=client_train["target"],
    random_state=42,
)

print("train  :", train_clients.shape, f"fraud rate {train_clients['target'].mean():.2%}")
print("holdout:", holdout_clients.shape, f"fraud rate {holdout_clients['target'].mean():.2%}")

In [ ]:
# Invoices follow their client
train_invoices = invoice_train[invoice_train["client_id"].isin(train_clients["client_id"])]
holdout_invoices = invoice_train[invoice_train["client_id"].isin(holdout_clients["client_id"])]

print("train invoices  :", train_invoices.shape)
print("holdout invoices:", holdout_invoices.shape)
print("clients on both sides:", len(set(train_invoices["client_id"]) & set(holdout_invoices["client_id"])))

In [ ]:
# Save holdout client IDs so the whole team uses the same split
Path("data/split").mkdir(exist_ok=True)
holdout_clients[["client_id"]].to_csv("data/split/holdout_ids.csv", index=False)

## 2.2 Leakage check: end of the invoice history

### Why this check?

We don't know when a fraud was found (see the Stage 1 checklist). If STEG replaces the meter or corrects bills **after** finding a fraud,
fraud clients would look different at the **end** of their history – and the model could "cheat" by using that.

Big difference → warning sign for leakage. Small difference → good sign, but no proof.

In [ ]:
# Leakage check: do fraud clients look different at the end of their history? (train only)
inv = train_invoices.copy()
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"])

per_client = inv.groupby("client_id").agg(
    last_invoice=("invoice_date", "max"),
    n_counters=("counter_number", "nunique"),
).join(train_clients.set_index("client_id")["target"])

per_client.groupby("target").agg(
    median_last_invoice=("last_invoice", "median"),
    mean_n_counters=("n_counters", "mean"),
)

## 2.3 EDA: client table

In [ ]:
# District: value counts
print(train_clients["disrict"].value_counts())

In [ ]:
# Region: value counts, train vs. test
print(train_clients["region"].value_counts())
print(client_test["region"].value_counts())
# print(train_invoices.duplicated().sum())     #drop duplicated rows

In [ ]:
# Regions per district
train_clients.groupby("disrict")["region"].unique().apply(sorted)

In [ ]:
# Regions that appear in only one of the two files
train_regions = set(train_clients["region"])
test_regions = set(client_test["region"])

print("only in train:", sorted(train_regions - test_regions))
print("only in test :", sorted(test_regions - train_regions))

## 2.4 EDA: consumption levels

In [ ]:
# Invoice table: summary statistics
train_invoices.describe().T

In [ ]:
# Consumption levels: overview (train only)
levels = ["consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4"]

summary = train_invoices[levels].describe(percentiles=[0.5, 0.9, 0.99]).T
summary["share_zero"] = (train_invoices[levels] == 0).mean()
summary

In [ ]:
# Distribution of consumption level 1 (log scale, train only)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_invoices["consommation_level_1"].plot.hist(bins=100, ax=axes[0], title="Level 1 (raw)")
np.log1p(train_invoices["consommation_level_1"]).plot.hist(bins=100, ax=axes[1], title="Level 1 (log1p)")
plt.show()

In [ ]:
# Mean consumption per client, fraud vs. no fraud (train only)
per_client = (
    train_invoices.groupby("client_id")[levels].mean()
    .join(train_clients.set_index("client_id")["target"])
)

display(per_client.groupby("target")[levels].median())

sns.boxplot(data=per_client, x="target", y=np.log1p(per_client["consommation_level_1"]))
plt.title("Mean consumption level 1 per client (log1p) by target")
plt.show()

In [ ]:
# Check: does the sum of the 4 levels equal new_index - old_index? (train only)
levels = ["consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4"]
total = train_invoices[levels].sum(axis=1)
index_diff = train_invoices["new_index"] - train_invoices["old_index"]

print("share of invoices where sum == index difference:", (total == index_diff).mean().round(3))

In [ ]:
# Mismatch between levels and index difference: more common for fraud? (train only)
mismatch = train_invoices.assign(mismatch=(total != index_diff))

mismatch_per_client = (
    mismatch.groupby("client_id")["mismatch"].any()
    .to_frame()
    .join(train_clients.set_index("client_id")["target"])
)

print(mismatch_per_client.groupby("target")["mismatch"].mean().round(3))

## 2.5 EDA: counter information

In [ ]:
# Is counter_type a useful feature? Fraud rate per type combination (train only)
types_per_client = (
    train_invoices.groupby("client_id")["counter_type"]
    .agg(lambda s: "+".join(sorted(s.unique())))   # e.g. "ELEC", "GAZ", "ELEC+GAZ"
    .rename("types")
    .to_frame()
    .join(train_clients.set_index("client_id")["target"])
)

types_per_client.groupby("types")["target"].agg(n_clients="count", fraud_rate="mean")

In [ ]:
# reading_remarque: how often, and how much fraud per code? (train only)
remarque = (
    train_invoices[["client_id", "reading_remarque"]]
    .merge(train_clients[["client_id", "target"]], on="client_id")
    .groupby("reading_remarque")["target"]
    .agg(n_invoices="count", fraud_rate="mean")
    .sort_values("n_invoices", ascending=False)
)
remarque

### Decision: reading_remarque

- Codes 6, 8 and 9 make up 99.98 % of all bills, and their fraud rate is almost the same (7.6 / 8.1 / 8.0 %, counted per bill). So the code does not help to tell fraud apart.
- Code 7: only 515 bills. Codes 203, 413, 207, 5: just 28 bills together – probably typing errors (these numbers look like `counter_code` values).

**Decision:** DROP the whole column – no useful signal. (If we kept it: set codes > 100 to NaN = "unknown", instead of deleting the bills.)

## 2.6 EDA: data errors (months_number, creation_date)

In [ ]:
# months_number: where do the extreme values come from? (train only)
print(train_invoices["months_number"].describe(percentiles=[0.5, 0.9, 0.99, 0.999]))

top = train_invoices.nlargest(29, "months_number")[
    ["client_id", "invoice_date", "months_number", "consommation_level_1", "counter_type"]
].merge(train_clients[["client_id", "disrict", "region", "target"]], on="client_id")
top
print((train_invoices["months_number"]>500).value_counts())

In [ ]:
# How many clients have at least one invoice with months_number > 600?
print(train_invoices.loc[train_invoices["months_number"] > 600, "client_id"].nunique())
# months_number: how many invoices above typical billing periods? (train only)
for limit in [0, 1, 4, 12, 24, 48, 60, 240, 480, 1200, 4800, 6000]:
    print(f"> {limit:>3} months: {(train_invoices['months_number'] > limit).sum():>7} invoices")


# creation_date: convert text (DD/MM/YYYY) to real dates first
creation = pd.to_datetime(train_clients["creation_date"], dayfirst=True)
print(creation.min(), creation.max())

### Decision: months_number

`months_number` = how many months one bill covers.

- Normal: median 4 months; 99 % of the bills cover ≤ 12 months, 99.9 % ≤ 24.
- Above 240 months (20 years) the count hardly changes: 1,263 bills > 240 months, still 1,238 > 6,000 months (500 years!). Maximum: 636,624 months ≈ 53,000 years.
  These are **impossible values** (data errors), and they come from only **38 clients**. There are also 2 bills with 0 months.
- The normal-distribution rule (mean ± 2 standard deviations) does not work here: the data is very lopsided (skewed), and the errors make the standard deviation huge (≈ 3,400). The limit would be ≈ 6,800 months – so most errors would count as "normal".
- A check using `creation_date` did not work either (see *Decision: creation_date*).

**Decision:** new yes/no column `months_error = 1` if `months_number > 240`. The wrong value itself gets replaced (NaN or a maximum), so features like "consumption per month" are not destroyed. In Stage 6 we see if the model finds this flag useful.

In [ ]:
# Plausibility rule: an invoice can't cover more months than the client exists (train only)
inv = train_invoices[["client_id", "invoice_date", "months_number"]].merge(
    train_clients[["client_id", "creation_date"]], on="client_id"
)
inv["invoice_date"] = pd.to_datetime(inv["invoice_date"])
inv["creation_date"] = pd.to_datetime(inv["creation_date"], dayfirst=True)

months_alive = (inv["invoice_date"] - inv["creation_date"]).dt.days / 30.44
implausible = inv["months_number"] > months_alive

print("implausible invoices:", implausible.sum())
print("affected clients    :", inv.loc[implausible, "client_id"].nunique())

In [ ]:
# How many invoices are dated BEFORE the client was created? (train only)
print("invoices before creation_date:", (months_alive < 0).sum())
print("affected clients             :", inv.loc[months_alive < 0, "client_id"].nunique())

### Decision: creation_date

- `creation_date` is stored as text (DD/MM/YYYY). To work with dates we convert it with `pd.to_datetime(..., dayfirst=True)`. Train range: 1977-02-05 to 2019-09-10.
- Idea: a bill can't cover more months than the client exists. This rule flags 59,951 bills from 40,221 clients (≈ 37 % of all train clients) – far more than the ~1,240 impossible values.
- 10,873 bills from 6,931 clients are dated **before** the client was created. That is impossible – so `creation_date` is not the real start date (probably the date the current contract was set up).

**Decision:** `creation_date` is not reliable → DROP (team decision).

## 2.7 EDA: additional investigations
 

In [ ]:
# Fraud rate per category for the open columns (train only)
def fraud_rate_by(series):
    return (series.to_frame("value").join(train_clients.set_index("client_id")["target"])
            .groupby("value")["target"].agg(n_clients="count", fraud_rate="mean")
            .sort_values("n_clients", ascending=False))

client_cols = ["disrict", "client_catg"]
invoice_cols = ["tarif_type", "counter_statue", "counter_code", "counter_coefficient"]

for col in client_cols:
    print(f"\n===== {col}")
    display(fraud_rate_by(train_clients.set_index("client_id")[col]))

for col in invoice_cols:
    # one value per client: the most common one in their invoices
    values = train_invoices[col].astype(str) if col == "counter_statue" else train_invoices[col]
    most_common = values.groupby(train_invoices["client_id"]).agg(lambda s: s.mode().iloc[0])
    print(f"\n===== {col} (most common per client)")
    display(fraud_rate_by(most_common))

### Decision: categorical columns

For every value we count the clients and their fraud rate (train only; for bill columns we take the most common value per client). Average fraud rate: 5.6 %.

**Careful with small groups:** in a tiny group, 0 % fraud can happen by pure chance. Rule of thumb: if `n_clients × 5.6 %` is below ~5 (fewer than 5 expected fraud cases), don't trust it → put such values into "other".

| Column | Finding | Decision |
| --- | --- | --- |
| `disrict` | 60: 3.6 %, 62: 5.2 %, 63: 6.5 %, 69: 7.1 % | DROP for the baseline model (no clear signal seen) |
| `client_catg` | 11: 5.5 % (105,216), 12: 4.0 % (1,879), **51: 16.9 %** (1,299) | feature, extra flag for 51 (3× the average) |
| `tarif_type` | 11: 5.3 % (91,269); all other values together ≈ 7.3 % (17,122) | feature, extra flag for "not 11" |
| `counter_statue` | 0: 5.6 % (107,337), **1: 1.1 %** (805); 2–5: 0 % (250 clients); 269375 / 618 invalid | feature with the groups 0 / 1 / other – only 0 and 1 contain fraud cases |
| `counter_code` | e.g. 207: 2.3 % (15,870), 5 / 202: ~7 % | feature later for combinations / fraud score, too complex for the baseline |
| `counter_coefficient` | 1 for 99.97 % of the clients; other values only 28 clients | DROP (almost always the same value) |

## 2.8 Correlation matrix

In [ ]:
# Correlation matrix of client-level feature candidates (train only)
levels = ["consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4"]
inv = train_invoices.assign(
    total=train_invoices[levels].sum(axis=1),
    mismatch=train_invoices[levels].sum(axis=1) != (train_invoices["new_index"] - train_invoices["old_index"]),
)

candidates = inv.groupby("client_id").agg(
    n_invoices=("total", "size"),
    total_mean=("total", "mean"),
    mismatch_share=("mismatch", "mean"),
    n_counters=("counter_number", "nunique"),
    has_both=("counter_type", lambda s: s.nunique() == 2),
).join(train_clients.set_index("client_id")[["client_catg", "target"]])
candidates["catg_51"] = candidates.pop("client_catg") == 51

corr = candidates.astype(float).corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix of feature candidates (train, client level)")
plt.show()

### Decision: counter_type & correlation findings

Correlation (from −1 to 1) shows how strongly two columns move together in a straight line. 0 = no straight-line connection.

| Feature | Correlation with `target` |
| --- | --- |
| `n_counters` | 0.19 |
| `n_invoices` | 0.12 |
| `has_both` (ELEC+GAZ) | 0.06 |
| `catg_51` | 0.05 |
| `total_mean` | 0.02 |
| `mismatch_share` | 0.00 |

- `n_counters` ↔ `has_both` = 0.75: closely connected (electricity + gas means at least two meters), but below 0.9 – so not exactly the same information. `n_counters` is the stronger signal.
- `n_invoices` ↔ `n_counters` = 0.56: long-time customers have more bills and more meter changes.
- No pair above 0.9 → we don't have to remove a column because two columns say the same thing (multicollinearity).
- Correlation misses rare but strong signals: `catg_51` has 3× the fraud rate but only 0.05, because just 1.2 % of the clients are in category 51.
- `mismatch_share` is 0.00, but "at least one mismatch" showed 8.9 % vs. 5.7 % → the yes/no flag is the better feature.

**Decisions:**
- `counter_type`: DROP – `has_both` is closely connected to `n_counters` (0.75) and the weaker signal (0.06 vs. 0.19).
- Mismatch: use the yes/no flag "at least one mismatch" instead of the share.

## 2.9 EDA by Luca (PR #19): investigation & cleaning

### Note: EDA by Luca (merged from PR #19)

Luca's investigation from the team notebook on `main`, merged into this notebook so nobody's work gets lost.
- `train_merged` = bills + client data in one table, for analyses that need all columns.
- **Careful:** the `counter_statue` conclusion **deletes** bills with invalid status values in train and holdout. This runs before Stage 3, so the feature table is built from the cleaned bills. Stage 3 originally grouped them into "other" instead – see *Differences to this notebook* in 3.2.

In [ ]:
#create merged training dataset for when every columns is needed during analysis
train_merged = train_invoices.merge(train_clients, on="client_id", how="inner")

### counter_statue

#### Analysis

`counter_statue` has a few invalid values, should be only between 0 and 5.

The amount is quite low but let's check first if it is somehow related to the target outcome first.

Do the decision on training data.

In [ ]:
valid_values = ["0", "1", "2", "3", "4", "5"]
invalid_invoice = train_invoices[~train_invoices["counter_statue"].isin(valid_values)]

print(f"Number of invalid rows {invalid_invoice.shape[0]}")
merged_invalid_rows = pd.merge(train_clients, invalid_invoice, on="client_id", how="inner")


individual_clients = merged_invalid_rows["client_id"].value_counts().count()
fraudulent = merged_invalid_rows["client_id"][merged_invalid_rows["target"] != 0].count()

print(f"By deleting rows with invalid 'counter_statue' we lose data from {individual_clients} individual clients and {fraudulent} fraudulent entries")



#### Conclusion

It is probably safe to just delete those columns since the impact is minor and does not further delete positive classes.

Delete in both splits.

In [ ]:
train_invoices = train_invoices[train_invoices["counter_statue"].isin(valid_values)]
holdout_invoices = holdout_invoices[holdout_invoices["counter_statue"].isin(valid_values)]
train_merged = train_merged[train_merged["counter_statue"].isin(valid_values)]

train_invoices["counter_statue"].value_counts()

### months_number

#### Analysis

Does this reporesent the 12 months?

In [ ]:
train_invoices["months_number"].value_counts().head(20)

In [ ]:
rows_with_invalid_months = train_invoices[~train_invoices["months_number"].between(1,12)]
invalid_months = rows_with_invalid_months["months_number"].value_counts()
invalid_months_ratio = 100 * rows_with_invalid_months.size / train_invoices.shape[0]
print(f"There are  {rows_with_invalid_months.size} invalid month entries, ratio: {invalid_months_ratio:.2f} %\n")
print(invalid_months)

In [ ]:
tmp = pd.concat([pd.to_datetime(train_invoices["invoice_date"]).dt.month, train_invoices["months_number"]], axis=1)
display(tmp.head())
tmp[["invoice_date", "months_number"]].corr()

Probably not the month of the invoice.

Let's see if it is the number of months since the last invoice, so the number of months in which the consumption occurred.

In [ ]:
# Months between consecutive invoices of the same client (train only)
invoice_gaps = train_merged.copy()
correlation_matrix = invoice_gaps[["months_number", "target"]].corr()
invoice_gaps["invoice_date"] = pd.to_datetime(invoice_gaps["invoice_date"])
invoice_gaps = invoice_gaps.sort_values(["client_id", "invoice_date"]).reset_index(drop=True)

# diff() within each client -> first invoice of every client gets NaT (no previous row)
# do it raw nad also sort by counter_type (GAZ/ELEC)
days_since_last_raw = invoice_gaps.groupby(["client_id"])["invoice_date"].diff().dt.days
days_since_last_sort = invoice_gaps.groupby(["client_id", "counter_type"])["invoice_date"].diff().dt.days
invoice_gaps["months_since_last_raw"] = (days_since_last_raw / 30.44).round()
invoice_gaps["months_since_last_sort"] = (days_since_last_sort / 30.44).round()

# Keep only rows where the difference was calculated (drops each client's first invoice)
invoice_gaps = invoice_gaps.dropna(subset=["months_since_last_raw", "months_since_last_sort"])
invoice_gaps["months_since_last_raw"] = invoice_gaps["months_since_last_raw"].astype(int)
invoice_gaps["months_since_last_sort"] = invoice_gaps["months_since_last_sort"].astype(int)

difference_raw = np.absolute(invoice_gaps["months_since_last_raw"] - invoice_gaps["months_number"])
difference_sort = np.absolute(invoice_gaps["months_since_last_sort"] - invoice_gaps["months_number"])

print(correlation_matrix)

In [ ]:
difference_raw = np.absolute(invoice_gaps["months_since_last_raw"] - invoice_gaps["months_number"])
difference_sort = np.absolute(invoice_gaps["months_since_last_sort"] - invoice_gaps["months_number"])
ratio_raw = 100 * (difference_raw <= 1).sum() / difference_raw.size
ratio_sort = 100 * (difference_sort <= 1).sum() / difference_sort.size
print("Ratio of rows where months_number is the time since the last invoice +/- 1 month:")
print(f"Sorted only by client {ratio_raw:.2f}")
print(f"Sorted also by GAZ/ELEC {ratio_sort:.2f}")

#### Conclusion

It is not clear what this columns says, since
1. The top entries are not nicely in the 1 - 12 range
2. No correlation with the invoice date
3. No strong evidence that it is the months since the last invoice (duration of consumption)

### Consumption levels, new and old index

#### Analysis

Let's see how consumption and counter indices are related

In [ ]:
# is index difference same as sum of consumptions?
sum_of_conumption = train_invoices[[f"consommation_level_{i}" for i in range(1, 5)]].sum(axis=1)

mismatch = sum_of_conumption != (train_invoices["new_index"] - train_invoices["old_index"])
mismatch_ratio = 100 * mismatch.sum() / train_invoices.shape[0]

print(f"The mismatch ratio is {mismatch_ratio:.2f} %")

In [ ]:
# how high is fraudulence in the mismatch group
# get client_id where mismatch happened
mismatched_clients = train_invoices["client_id"][mismatch].unique()
# sum up the frauds 
sum_of_frauds = train_clients["target"][train_clients["client_id"].isin(mismatched_clients)].sum()

# calculate the fraudulence ratio between all clients with mismatches
fraud_ratio_mismatch = 100 * sum_of_frauds / mismatched_clients.size
print(f"The group that had mismatches has a fraudulence rate of {fraud_ratio_mismatch:.2f} %")

In [ ]:
# are the consumption levels mutually exclusive as expected?
# make a sample so the plot is quicker. The sample should keep same fraudulence ratio
plot_sample = train_merged.groupby("target", group_keys=False).sample(frac=0.1)

levels = [f"consommation_level_{i}" for i in range(1, 5)]
# still takes ~3m, so uncomment if neccessary
#sns.pairplot(plot_sample[levels + ["target"]], hue="target", vars=levels, plot_kws={"alpha": 0.3})

#### Conclusion

In the client group where sum of consumptions does not match the difference of the index the fraudulence ratio is higher, so it seems to be a good idea to create a "mismatch" feature to capture this.

The consuption levels are not mutually exclusive.

Fraudulence seems to be prevalent in all the levels and in the lower value range. But the ranges are not the same, so we cannot simply add the values.

There are strong outliers in the levels, so some kind of preprocessing should be done when using reggression models.

### Counter_type

#### Analysis

Is the fraudlunce going on predominantly on Gas or Electricity?

In [ ]:
train_temp = train_merged[["target", "counter_type"]]
train_temp["counter_type"] = train_temp["counter_type"].map({"ELEC":0,"GAZ":1})
train_temp.corr()

#### Conclusion

No correlation, so fraud is going on on both.

### disrict / region

#### Analysis

Let's see if and how it is related to region.

In [ ]:
districts_by_region = (
    train_clients.groupby("disrict")["region"]
    .unique()
    .apply(sorted)
)
display(districts_by_region)


In [ ]:
counts = train_clients.groupby(["region", "target"]).size().unstack(fill_value=0)
fraud_ratio = counts[1] / counts[0]

fig, ax = plt.subplots(figsize=(10, 5))
fraud_ratio.plot(kind="bar", ax=ax)
ax.set_xlabel("Region")
ax.set_ylabel("Frauds / non-frauds")
ax.set_title("Fraud to non-fraud ratio per region")
plt.tight_layout()
plt.show()

In [ ]:
counts = train_clients.groupby(["disrict", "target"]).size().unstack(fill_value=0)
fraud_ratio = counts[1] / counts[0]

fig, ax = plt.subplots(figsize=(10, 5))
fraud_ratio.plot(kind="bar", ax=ax)
ax.set_xlabel("District")
ax.set_ylabel("Frauds / non-frauds")
ax.set_title("Fraud to non-fraud ratio per destrict")
plt.tight_layout()
plt.show()

#### Conclusion

Regions (with a few examples) seem to belong to one specific district.

This means Region is a more granular separation of location.

There are areas where fraud is double than others.

It has to be disussed if it is ethically acceptable to profile people to where they come from.

# ✅ Stage 2 End: Results & Learnings (Marker)

## 2.10 Split decisions & EDA findings

### Rubber-duck checklist

- [x] **Was the test set split off before any deeper analysis/scaling?** → Yes: 20 % holdout taken from `client_train` and put into `train_clients`, split by client, before any analysis.
- [x] **Is there a time dimension in the data?** → Invoices have dates, but the `target` has none (one answer per client) → a random split by client is fine.
- [x] **Is the target class extremely imbalanced?** → Yes, only 5.6 % fraud → `stratify=y` keeps the share equal in both parts.

### EDA findings (train only)

- **No missing values.** The problems are impossible or extreme values instead.
- **`months_number`:** impossible values (> 240 months) at 38 clients → flag `months_error` + replace the value.
- **`creation_date`:** not reliable (bills before the client existed) → DROP.
- **Consumption:** levels 1–4 add up to `new_index − old_index` in 99.6 % of the bills. Clients with at least one bill where it doesn't add up: 8.9 % fraud vs. 5.7 %.
- **`counter_type`:** electricity + gas 7.2 % fraud vs. electricity only 4.3 % (gas only: 554 clients – too few to say).
- **`n_counters`:** fraud clients have more meters (2.31 vs. 1.67) → good candidate, but maybe leakage (meter changed after the fraud was found).
- **`region`:** region 199 only in train; no region that is only in the Zindi test.
- **`client_catg` 51:** 16.9 % fraud (3× the average). **`tarif_type` not 11:** ≈ 7.3 % vs. 5.3 %.
- **`counter_statue` 1:** 1.1 % fraud (5× below average); 269375 / 618 are invalid values.
- **`counter_code` 207:** 2.3 % fraud at 15,870 clients. `counter_coefficient` is almost always 1.
- **Small groups** with 0 % fraud are chance (`n × 5.6 %` < ~5) → group them into "other", no fixed rules.
- **Correlation:** strongest link to `target`: `n_counters` 0.19, `n_invoices` 0.12. No pair above 0.9. Rare strong signals look weak in a correlation (`catg_51`: 0.05).
- **Dropped:** `reading_remarque` (no signal), `counter_type` (already covered by `n_counters`), `counter_coefficient` (almost constant).

### Team decisions per column

Decisions from the team discussion (2026-10-20). Justifications follow in the Micro area of the related section.

**Client table (`client_train`)**

| Column | Type | Decision / note | Why |
| --- | --- | --- | --- |
| `disrict` | int | DROP // replace name "district" | 60: 3.6 % vs. 69: 7.1 % fraud; dropped for the baseline (2.6.2) |
| `client_id` | int | DROP | identifier, no information about behaviour |
| `client_catg` | int | dummy variables → feature | 51: 16.9 % fraud (1,299 clients) vs. 5.6 % → flag 51 (2.6.2) |
| `region` | int | correlation matrix → dummy variables → feature | 25 categories; 199 only in train (2 clients in the full data), none only in test (2.3) |
| `creation_date` | date | DROP | unreliable as start date: 10,873 invoices of 6,931 clients are dated before it (2.6) |
| `target` | int | drop from X and use it as `y` | label (1 = fraud), Stage 1 |

**Invoice table (`invoice_train`)**

| Column | Type | Decision / note | Why |
| --- | --- | --- | --- |
| `client_id` | int | DROP - no feature| link key to the client table, no information about behaviour |
| `invoice_date` | date | convert2 - feature | open (e.g. consumption trend over time); end of history similar for fraud / no fraud (2.2) |
| `tarif_type` | --- | investigate - correlation matrix | not 11: ≈ 7.3 % vs. 11: 5.3 % fraud → flag "not 11" (2.6.2) |
| `counter_number` | int | DROP - no feature | meter ID, no information as a number (count per client: see 2.2) |
| `counter_statue` | --- | investigate - correlation matrix (DROP invalid rows, 47!) | only 0 (5.6 %) and 1 (1.1 %) contain fraud; rest rare/invalid → 0 / 1 / other (2.6.2) |
| `counter_code` | --- | investigate - correlation matrix | e.g. 207: 2.3 % vs. 5 / 202: ~7 % → later for combinations / fraud score (2.6.2) |
| `reading_remarque` | int | feature (drop rows over 100 or write other) | 6 / 8 / 9 almost equal fraud rate, codes > 100 are errors → DROP whole column (2.5) |
| `counter_coefficient` | int | feature (sounds like a feature) | 1 for 99.97 % of clients → almost constant, DROP (2.6.2) |
| `consommation_level_1` – `_4` | int | convert2 - feature | per invoice → aggregate per client (one row per client); distributions see 2.4 |
| `old_index`, `new_index` | --- | covert2 - combine: `consumption_mismatch` = sum(level 1–4) ≠ new_index − old_index | sum of levels = index difference in 99.6 %; ≥ 1 mismatch: 8.9 % fraud vs. 5.7 % (2.4) → flag "at least one mismatch", share has correlation 0.00 (2.7) |
| `months_number` | --- | investigate - now DROP but convert2?? | impossible values > 240 months in 38 clients → flag `months_error`, replace value (2.6); periods differ (4 or 12 months) → consumption per month? |
| `counter_type` | --- | feature → dummy variable | ELEC+GAZ 7.2 % vs. ELEC 4.3 %, but `has_both` correlates 0.75 with `n_counters` (weaker: 0.06 vs. 0.19) → DROP (2.7) |

**Open points**
- Issue "Fraud Detection at 1 client over a bigger time period" (RandomSelection)
- Pipeline structure

### Open points (not handled yet)

- Is `consommation_level_3` only used for gas (or `consommation_level_2` only for electricity)? Idea from Karl, not checked yet.
- `history_years` is at most 14.9 years although single bills go back to 1977 – data quirk or old one-off bills?
- Team ideas for later (see 3.2): seasonal anomaly (Hadi), years as client, `disrict` / `region`, `counter_statue`, `months_error`.

## 2.11 Learnings (Marker)

### Stage 4: Marker – Fraud score as rule-based baseline (scorecard) & feature combinations

- One column alone is often weak; **combined** with others it can be strong → **interaction features** (new columns built from several columns, e.g. `client_catg` 51 × electricity+gas × `n_counters`).
- Idea: give points for every warning sign → add them up to a `fraud_score` per client → above a threshold = "suspicious". That is a simple rule-based baseline for issue #10 (a scorecard).
- Fair test: decide the points on the training folds only and test them on the validation fold. Otherwise the score already "knows" the answers (leakage).
- Tree models (LightGBM) find some combinations by themselves; LogReg and the scorecard need them as ready-made columns. Ready-made columns are also easier to explain to STEG.

# ✅ Stage 3: Cleaning & Feature Preprocessing

### In short

The model needs **one row per client**, but each client has ~33 bills → `build_features` summarises the bills into 18 features, e.g. average consumption, how unsteady it is (`total_std`), share of bills with zero consumption, flags for errors and mismatches. The same function runs for train, holdout and Zindi test.

- Steps that **learn from the data** (median for missing values, scaling) sit in a **pipeline** – a fixed chain of steps that is fitted on training data only → no leakage.
- 3.2 documents the team call and where the team differs from this notebook.

**→ Next:** set the simplest possible scores that every later model has to beat.

## 3.1 Preprocessing rules: fixed rule or learned (fit)?

### Fixed rules vs. learned steps

A **fixed rule** works the same for every client and uses only that client's own data – it can run before the split without leakage.
A **learned step** calculates something from many clients (a median, a mean, a standard deviation) → `fit()` on train only, then `transform()` on holdout/test. These go into a sklearn `Pipeline`.

| Step | Type | Why |
| --- | --- | --- |
| `months_number > 240` → NaN | fixed rule | same limit for everyone |
| NaN → median | **fit** | the median is calculated from the data → train only |
| rare `counter_statue` values → "other" | fixed rule | we chose the groups 0 / 1 / other ourselves |
| features per client (e.g. mean consumption) | fixed rule | uses only the client's own bills |
| `StandardScaler` | **fit** | mean and spread come from the data → train only |
| `log1p` | fixed rule | same formula for every value |

- Exception: as soon as a feature uses **other clients** (e.g. "consumption vs. region average", fraud rate per category = target encoding), it is a learned step.
- IDs (`client_id`, `counter_number`) are not features: the model would learn the numbers by heart (overfitting). Counts made from them (`n_counters`) are fine.
- `log1p` keeps the order of the values, so trees (Random Forest, LightGBM) don't need it; Logistic Regression does.

## 3.2 Team discussion (Wednesday, 2026-10-21)

### Team discussion from 13:30–16:00


**Agreed (matches this notebook)**
- The Zindi test set is only for the submission; we train and evaluate on our own split of `client_train`. The Zindi leaderboard score is added in Stage 7 as an outside comparison (Mohamed).
- No time-based split: one label per client, and we don't know when the fraud started (Hadi).
- The bills of a client are not independent rows but a kind of time series → condense them into 

**one row per client** with features (Luca, Hadi).
- `mismatch` flag: levels 1–4 ≠ `new_index − old_index` → higher fraud rate (Luca).
- `client_catg` 51 is the clearest fraud signal (Mohamed's plot). Hadi: the categories are probably private / business / industrial or administration.
- `reading_remarque`: codes 6, 8, 9 have almost the same fraud rate, code 7 has too few bills (~500) → not used.

**Workflow**
- Personal notebooks stay local and are not pushed (Luca); Karl merges agreed parts into the team notebook ("secretary").
- Agile: build the baseline with the main features first; uncertain features get a "?" and are checked later (Mohamed).
- Before closing Stage 3, the team's feature list has to be compared with this notebook (Luca). Proposal (Karl, Mohamed): baseline first with the current features, then compare with the team's feature set.

### Differences to this notebook

| Topic | This notebook | Team (2026-10-21) | Status |
| --- | --- | --- | --- |
| `disrict` | DROP for the baseline | **use it** (vote 3:1); idea: combine `disrict` + `region` into one "address" column (Mohamed; regions lie inside districts). Luca raised the ethics question whether clients may be judged by their area | **decided (Karl):** baseline without `disrict`, then compare a model with `disrict` in Stage 4/5 |
| `counter_statue` | features `statue_1`, `statue_other` (most common value) | **not used for now, "?"**: correlation ~0.004, and > 30 % of clients have changing status values across bills; Hadi wanted to keep it (mode) | open |
| `creation_date` | DROP (unreliable start date) | **try later**: newer clients seem to cheat less (modern meters?) → "years as client" (Luca), flag "bill before creation date" (Hadi) | "?" – later |
| `invoice_date` | only `history_years` | **seasonal anomaly**: consumption variation per season compared with the district, as one column (Luca, Hadi) | Hadi is working on it |
| `months_error` | flag at `months_number > 240` | disputed: Luca saw no meaning for it in his EDA | open |
| Variation per client | `total_std`, `total_max` | Hadi: **range ÷ mean** (relative, so a bakery and a flat are comparable) | idea |
| `tarif_type` | flag "not 11" | Mohamed: 45 has the highest fraud rate (here: 12.2 % on 131 clients, already inside "not 11") | covered |
| `counter_statue` invalid values | grouped into "other" | **Luca (PR #19, section 2.9): delete the bills** with invalid status (minor impact, no positive classes lost) – active now, runs before Stage 3 | decided by Luca's merged cleaning |
| `counter_statue` 5 | 0 % fraud as most common value (65 clients) | Mohamed's plot: 5 has the highest fraud rate (counted per bill, not per client) | check counting method |

## 3.3 Feature table: one row per client

### Feature plan (from the Stage 2 decisions)

| Feature | Built from | Stage 2 decision |
| --- | --- | --- |
| `region` | `region` | stays in the table, **not in the baseline** – comes back later (Stage 5, together with `disrict`) |
| `catg_51` | `client_catg == 51` | feature, flag for 51 |
| `n_invoices` | number of bills | correlation 0.12 with `target` |
| `n_counters` | different `counter_number` | strongest correlation (0.19) |
| `level_1_mean` … `level_4_mean` | `consommation_level_1–4` | aggregate per client |
| `total_mean`, `total_std` | sum of the 4 levels per bill | aggregate per client (`total_max` dropped in Stage 4: correlation 0.907 with `total_std`) |
| `zero_share` | share of bills with 0 consumption | new idea: manipulated meters show 0 |
| `total_per_month_mean` | total / `months_number` (cleaned) | months differ per bill (4 or 12) |
| `months_error` | any bill with `months_number > 240` | flag |
| `mismatch_any` | any bill with levels ≠ `new_index − old_index` | flag instead of share |
| `history_years` | first → last `invoice_date` | `invoice_date`: convert to feature |
| `tarif_not_11` | most common `tarif_type` ≠ 11 | flag |
| `statue_1`, `statue_other` | most common `counter_statue` | groups 0 / 1 / other |

**Not used:** `client_id`, `counter_number` (IDs), `disrict` (baseline), `creation_date`, `reading_remarque`, `counter_type`, `counter_coefficient`.
**Later:** `counter_code` (combinations / fraud score), consumption trend over time.

In [ ]:
# Feature table: one row per client (fixed rules only, no fit)
LEVELS = ["consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4"]


def most_common(invoices, col):
    """Most common value of col per client; on a tie the smaller value wins (same as .mode())."""
    counts = invoices.groupby(["client_id", col]).size().rename("n").reset_index()
    top = counts.sort_values(["n", col], ascending=[False, True]).drop_duplicates("client_id")
    return top.set_index("client_id")[col]


def build_features(clients, invoices):
    inv = invoices.copy()
    inv["invoice_date"] = pd.to_datetime(inv["invoice_date"])
    inv["counter_statue"] = inv["counter_statue"].astype(str)
    inv["total"] = inv[LEVELS].sum(axis=1)
    inv["zero"] = inv["total"] == 0
    inv["mismatch"] = inv["total"] != inv["new_index"] - inv["old_index"]
    inv["months_error"] = inv["months_number"] > 240
    months = inv["months_number"].where((inv["months_number"] > 0) & ~inv["months_error"])
    inv["total_per_month"] = inv["total"] / months

    agg = inv.groupby("client_id").agg(
        n_invoices=("total", "size"),
        n_counters=("counter_number", "nunique"),
        level_1_mean=("consommation_level_1", "mean"),
        level_2_mean=("consommation_level_2", "mean"),
        level_3_mean=("consommation_level_3", "mean"),
        level_4_mean=("consommation_level_4", "mean"),
        total_mean=("total", "mean"),
        total_std=("total", "std"),
        zero_share=("zero", "mean"),
        total_per_month_mean=("total_per_month", "mean"),
        months_error=("months_error", "max"),
        mismatch_any=("mismatch", "max"),
        first_invoice=("invoice_date", "min"),
        last_invoice=("invoice_date", "max"),
    )
    agg["history_years"] = (agg.pop("last_invoice") - agg.pop("first_invoice")).dt.days / 365.25

    cl = clients.set_index("client_id")
    feats = cl[["region"]].copy()
    feats["catg_51"] = cl["client_catg"] == 51
    feats = feats.join(agg)
    feats["tarif_not_11"] = most_common(inv, "tarif_type") != 11
    statue = most_common(inv, "counter_statue")
    feats["statue_1"] = statue == "1"
    feats["statue_other"] = ~statue.isin(["0", "1"])

    bool_cols = feats.select_dtypes("bool").columns
    feats[bool_cols] = feats[bool_cols].astype(int)
    if "target" in cl:
        feats["target"] = cl["target"].astype(int)
    return feats

In [ ]:
# Build the feature tables for train, holdout and the Zindi test
train_features = build_features(train_clients, train_invoices)
holdout_features = build_features(holdout_clients, holdout_invoices)
test_features = build_features(client_test, invoice_test)

# Self-check: one row per client, same columns everywhere
for name, feats, clients in [("train", train_features, train_clients),
                             ("holdout", holdout_features, holdout_clients),
                             ("test", test_features, client_test)]:
    assert feats.index.is_unique and len(feats) == len(clients), name
    print(f"{name:8s} {feats.shape}")
assert list(train_features.columns.drop("target")) == list(test_features.columns)

In [ ]:
# Feature table: first look (train only)
display(train_features.head())
train_features.describe().T

## 3.4 Learned steps: preprocessing pipeline

### Why a pipeline?

The pipeline bundles the learned steps (median, scaler). In Stage 4/5 it is fitted **only on train / the train folds** and then applied to the validation data – so no information from validation or holdout leaks in.
- **Logistic Regression:** median imputation → `log1p` → `StandardScaler`. (`region` is left out of the simple baseline; when it comes back it needs one-hot encoding.)
- **Trees (Random Forest, LightGBM):** no scaling and no `log1p` needed. LightGBM can handle NaN itself; Random Forest needs the median imputation.

In [ ]:
# Preprocessing pipeline for Logistic Regression (fit later on train / train folds only)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

X_train = train_features.drop(columns=["target", "region"])  # region comes back later (Stage 5)
y_train = train_features["target"]

logreg_preprocessor = make_pipeline(SimpleImputer(strategy="median"), FunctionTransformer(np.log1p), StandardScaler())

# Check only: fit on train, look at the output shape
print("X_train:", X_train.shape, "-> after preprocessing:", logreg_preprocessor.fit_transform(X_train).shape)

# ✅ Stage 3 End: Results & Learnings (Marker)

## 3.5 Checklist & results

### Rubber-duck checklist

- [x] **Is every preprocessing step fitted on train only (`fit()` on train, `transform()` on test)?** → Yes. Fixed rules (limits, "other" groups, features per client from the client's own bills) run for every table the same way. Learned steps (median imputation, `StandardScaler`) sit in `logreg_preprocessor` and are fitted on train / train folds only.
- [x] **Are there IDs, timestamps or random index columns in the features?** → `client_id` and `counter_number` are IDs → not used (the model would learn the numbers by heart = overfitting). Only counts made from them (`n_counters`). Dates only as derived feature (`history_years`).
- [x] **Are there numeric features with extreme skewness?** → Yes, consumption (e.g. `total_std` up to 163,788 with median 326) → `log1p` for Logistic Regression. Trees don't need it, because `log1p` keeps the order of the values.

### Results

- **Feature tables:** train 108,394 × 18 features (+ `target`; the baseline uses 17, without `region`), holdout 27,099, Zindi test 58,069 – one row per client, same columns everywhere.
- **Flags match the Stage 2 analysis:** `tarif_not_11` 17,125 clients (7.3 % fraud), `statue_1` 805, `statue_other` 250, `catg_51` 1,299.
- **NaN left for the pipeline:** `total_std` (3,375 clients with only 1 bill), `total_per_month_mean` (18 clients, only invalid months) → median imputation (learned step).
- **LogReg preprocessing:** 17 columns in, 17 out (no one-hot needed without `region`).
- **`history_years`:** no client has more than 14.9 years between first and last bill, although single bills go back to 1977 → to check in Stage 4 (data quirk or old one-off bills?).
- **Most common value on ties:** the smaller value wins (same as `.mode()`), otherwise ELEC+GAZ clients with equal numbers of tariff 11 and 40 bills would get a random value.
- **After Luca's cleaning (2.9):** 41 bills with invalid `counter_statue` deleted (5 clients, no fraud case) in train and holdout – the Zindi test bills are not deleted; their invalid values land in `statue_other`. Baseline scores unchanged.

## 3.6 Learnings (Marker)

# ✅ Stage 4: Baselines & Feature Selection

### In short

Three baselines, tested with 5-fold **cross-validation** (5 rounds: train on 4/5 of the train part, test on the other 1/5, then average):

**Dummy 0.500** (coin flip) → **Scorecard 0.670** (5 warning signs, weights calculated from the data) → **Logistic Regression 0.788**.

- **In plain words:** pick one fraud client and one honest client at random – the Logistic Regression ranks the fraud client higher in **79 %** of the cases.
- Kept simple on purpose: 17 features; `total_max` dropped (almost the same information as `total_std`); `region` left out for now.

**→ Next:** try stronger models – they have to beat 0.788.

## 4.1 Baselines

### Baselines (5-fold CV on train only)

- **Dummy (reference only):** same score for everyone → no ranking → AUC 0.5. The absolute lower limit.
- **Scorecard:** 5 warning signs, each with a weight calculated from the data (see below) – a tiny, fully explainable model.
- **Logistic Regression:** the simple, explainable baseline model (features below).

> **Leakage guard:** every score is a 5-fold CV on the **train part only**. Everything that learns from the data – the scorecard weights, the median and the scaler of the pipeline – is fitted on the 4 training folds of each round only. The holdout stays untouched until Stage 7.

In [ ]:
# CV setup and dummy baseline (5-fold CV on train only)
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

dummy_auc = cross_val_score(DummyClassifier(), X_train, y_train, cv=cv, scoring="roc_auc")
print(f"Dummy AUC: {dummy_auc.mean():.3f} ± {dummy_auc.std():.3f}")

### Scorecard: weights from the data

Nobody chooses the points by hand – a formula calculates them from the fraud rates:

**weight = ln( fraud rate WITH the signal / fraud rate WITHOUT the signal )**

| Ratio | Weight | Meaning |
| --- | --- | --- |
| > 1 (more fraud with the signal) | positive | pushes the client up the list |
| = 1 | 0 | signal tells nothing |
| < 1 (less fraud with the signal) | negative | pushes the client down |

- Example: `catg_51` has 16.9 % fraud vs. ~5.5 % without → ratio ~3.1 → weight ≈ +1.1. `statue_1`: 1.1 % vs. ~5.6 % → ratio ~0.2 → weight ≈ −1.6.
- The logarithm makes "twice as often" and "half as often" equally strong, just with opposite signs.
- **Score of a client** = sum of the weights of all signals the client has. Higher score = more suspicious.
- The weights are calculated **inside every CV round** from the 4 training folds only → no leakage.

In [ ]:
# Scorecard with weights derived from the data (fitted on the training folds only)
SIGNALS = {
    "catg_51": lambda X: X["catg_51"] == 1,
    "n_counters >= 2": lambda X: X["n_counters"] >= 2,
    "mismatch_any": lambda X: X["mismatch_any"] == 1,
    "tarif_not_11": lambda X: X["tarif_not_11"] == 1,
    "statue_1": lambda X: X["statue_1"] == 1,
}


def fit_weights(X, y):
    """Weight per signal = ln(fraud rate with signal / fraud rate without signal)."""
    return {name: np.log(y[flag(X)].mean() / y[~flag(X)].mean()) for name, flag in SIGNALS.items()}


def score(X, weights):
    return sum(w * SIGNALS[name](X) for name, w in weights.items())


score_auc = []
for fit, val in cv.split(X_train, y_train):
    weights = fit_weights(X_train.iloc[fit], y_train.iloc[fit])  # learned on 4 folds only
    score_auc.append(roc_auc_score(y_train.iloc[val], score(X_train.iloc[val], weights)))
score_auc = np.array(score_auc)

print("weights (whole train):", {k: round(float(v), 2) for k, v in fit_weights(X_train, y_train).items()})
print(f"Scorecard AUC: {score_auc.mean():.3f} ± {score_auc.std():.3f}")

### Features used by the Logistic Regression (17)

| Group | Features |
| --- | --- |
| Client | `catg_51` |
| Bills & counters | `n_invoices`, `n_counters`, `history_years` |
| Consumption | `level_1_mean`, `level_2_mean`, `level_3_mean`, `level_4_mean`, `total_mean`, `total_std`, `zero_share`, `total_per_month_mean` |
| Flags | `months_error`, `mismatch_any`, `tarif_not_11`, `statue_1`, `statue_other` |

**Not in the baseline:** `region` (later, Stage 5), `total_max` (dropped: correlation 0.907 with `total_std`).

In [ ]:
# Baseline model: Logistic Regression + baseline table (5-fold CV on train only)
logreg = make_pipeline(logreg_preprocessor, LogisticRegression(max_iter=1000))
logreg_auc = cross_val_score(logreg, X_train, y_train, cv=cv, scoring="roc_auc")

baselines = pd.DataFrame({
    "model": ["Dummy", "Scorecard (weights from data)", "Logistic Regression"],
    "auc_mean": [a.mean() for a in (dummy_auc, score_auc, logreg_auc)],
    "auc_std": [a.std() for a in (dummy_auc, score_auc, logreg_auc)],
}).round(3)
baselines

In [ ]:
# Plot: baseline comparison (CV AUC, mean ± std)
plot_data = baselines.sort_values("auc_mean").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.axvline(0.5, color="#52514e", linestyle="--", linewidth=1)
ax.errorbar(plot_data["auc_mean"], plot_data["model"], xerr=plot_data["auc_std"],
            fmt="o", markersize=8, color="#2a78d6", ecolor="#52514e", elinewidth=1, capsize=3)
for y, value in enumerate(plot_data["auc_mean"]):
    ax.text(value + 0.012, y, f"{value:.3f}", va="center", fontsize=9, color="#0b0b0b")

ax.set_xlim(0.45, 0.9)
ax.set_ylim(-0.6, len(plot_data) - 0.4)
ax.text(0.503, len(plot_data) - 0.5, "coin flip", color="#52514e", fontsize=9)
ax.set_xlabel("ROC-AUC (5-fold CV, train only)")
ax.set_title("Baselines: how well does each model rank fraud above no fraud?")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", color="#e5e4e0", linewidth=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 4.2 Multicollinearity check

In [ ]:
# Feature pairs with |correlation| > 0.9 (train only)
corr = X_train.corr().abs()
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
pairs[pairs > 0.9].sort_values(ascending=False)

# ✅ Stage 4 End: Results & Learnings (Marker)

## 4.3 Checklist & results

### Rubber-duck checklist

- [x] **Is there a dumb baseline (e.g. `DummyClassifier`)?** → Yes: AUC 0.500. It gives everyone the same score, so nobody ranks above anyone else (every pair is a tie) → coin flip. This is the absolute lower limit.
- [x] **Is there a simple, interpretable reference model (Logistic Regression)?** → Yes: AUC 0.788 – the operational baseline every Stage 5 model has to beat. XGBoost / AdaBoost / LightGBM belong to Stage 5, not to the baseline.
- [x] **Do features correlate with each other above 0.9 (multicollinearity)?** → One pair: `total_std` ↔ `total_max` = 0.907. Kept `total_std` (Karl): fraud shows as **unsteady consumption over many bills**, not as one single big bill (matches Mohamed's "inconsistency between invoices"). After dropping `total_max`: no pair above 0.9.

### Results

**Simple baseline:** 17 features, without `region` and `total_max` (Karl: as simple as possible; `region` comes back in Stage 5).

| Model | AUC (5-fold CV) | ± |
| --- | --- | --- |
| Dummy (reference) | 0.500 | 0.000 |
| Scorecard (weights from data) | 0.670 | 0.004 |
| **Logistic Regression** | **0.788** | 0.005 |

- **0.788 is the score every Stage 5 model has to beat.**
- Scorecard weights (whole train): `n_counters ≥ 2` +1.41, `catg_51` +1.13, `mismatch_any` +0.44, `tarif_not_11` +0.34, `statue_1` −1.61. The hand-picked points (+4 / +2 / +2 / +1 / −3) reached almost the same AUC (0.669), but without the leakage problem now.
- Also tried (removed to keep the baseline simple): Decision Tree depth 5 – 0.783, Linear Regression as a score – 0.783.
- With `region` (one-hot) the Logistic Regression reached 0.803 → `region` adds **+0.015** (2–3× the CV spread) → test again in Stage 5.
- Dropping `total_max` changed nothing (0.804 → 0.803 with `region`).

## 4.4 Learnings (Marker)

# 🔄 Stage 5: Model Exploration & CV

### In short (plan – issue #11)

**Goal:** find a model that beats the Logistic Regression (**AUC 0.788**) by more than the CV spread (± ~0.005 = the normal up-and-down between the 5 rounds).

1. **Same test setup as Stage 4:** `StratifiedKFold` (5 folds, `random_state=42`), train part only, holdout stays untouched.
2. **Compare at least 3 model families:** Random Forest (many decision trees vote together), AdaBoost and LightGBM (trees built one after another, each one fixes the mistakes of the previous ones). Because fraud is rare, give fraud cases more weight (`class_weight` / `scale_pos_weight`).
3. **Tune the best one:** try different settings (e.g. `RandomizedSearchCV`) – also only inside the CV on train.
4. **Add the "later" features one by one** and check whether the AUC really goes up: `region`, `disrict`, seasonal anomaly (Hadi), years as client, `counter_statue`, `months_error`.
5. **Result:** one table – model, AUC mean, AUC spread, notes.

**→ Next:** understand why the best model decides the way it does.

# 🔄 Stage 5 End: Results & Learnings (Marker)

# ⏳ Stage 6: Diagnostics, Plots & Interpretability

### In short (plan – issue #12)

**Goal:** understand *why* the best model decides and *where* it fails.

1. **Overfitting check:** compare the score on the training data with the CV score – a big gap means the model learned the training clients by heart.
2. **Feature importance / SHAP** (shows how much each feature pushes a client's score up or down): do the top features make sense? Check suspicious ones for leakage, e.g. `n_counters`, `mismatch_any` – was the counter changed or the bill corrected *after* the fraud was found?
3. **Choose a threshold for STEG:** the model gives probabilities, STEG needs a yes/no list – e.g. "inspect the top 5 % of clients" → Recall, Precision, F2 and the confusion matrix (right / wrong yes and no) at that threshold.
4. **Error analysis:** which fraud clients does the model miss, and what do they have in common?

**→ Next:** the final test and a clear answer for STEG.

# ⏳ Stage 6 End: Results & Learnings (Marker)

# ⏳ Stage 7: Business Alignment & Turning Points

### In short (plan – issues #14, #15, #16)

**Goal:** one honest final test, and a clear answer for STEG: is the model worth using?

1. **Final test – only once:** evaluate the best Stage 5/6 model on the untouched **holdout** (20 %). No tuning after this – otherwise the holdout becomes a practice exam and the score is too optimistic.
2. **Worth the complexity?** Compare with the baselines (Logistic Regression 0.788, scorecard 0.670). If the gain is small, take the simpler model (cheaper, easier to explain).
3. **Translate into business terms:** e.g. "if STEG inspects the top 5 % of clients, x % of all fraud cases are found and y % of the visits hit a fraudster".
4. **Data product** (slide): a monthly ranked inspection list for the field teams – who uses it, how often, which decision it supports.
5. **Zindi submission:** retrain on the full `client_train`, predict **probabilities** (`predict_proba`, not `predict`) for `client_test`, save `client_id,target` to `data/output/submission.csv`.
6. **Turning points:** write down what did not work and what we would try next (future work for the presentation).

**→ Result:** the presentation for STEG and our Zindi leaderboard score.

# ⏳ Stage 7 End: Results & Learnings (Marker)